# Data Generation for Synthetic RCT Diagnostics

This notebook explains how the project generates experiment summaries for an LLM-based diagnosis agent.

The generator does not simulate every user. Instead, it produces experiment-level aggregates that are realistic enough for teaching agentic diagnosis:

- group sizes
- pre-period means and standard deviations
- post-period means and standard deviations
- conversion rates
- optional segment summaries
- hidden ground-truth issue labels for evaluation

That design keeps the dataset compact and easy to inspect while still supporting meaningful failure modes.

## Failure modes injected by the generator

The generator can inject several issues deliberately:

- `srm`: group sizes deviate from the expected allocation
- `imbalance`: control and treatment differ before the experiment begins
- `low_power`: the sample is too small or the effect is too weak relative to noise
- `outliers`: the post-period variance becomes unusually large
- `simpsons_paradox`: segment-level effects and pooled effects disagree because the traffic mix shifts

Each synthetic experiment includes a hidden label list so we can later compare the agent's diagnosis to the truth.

In [ ]:
from rct_diagnosis_agent.data import SyntheticExperimentGenerator

generator = SyntheticExperimentGenerator(seed=7)

## A clean example

We start with a baseline experiment that does not force any specific failure mode.

In [ ]:
clean = generator.generate("exp_clean", failure_modes=[])
clean.model_dump()

## Targeted failure-mode examples

The next cell shows one example for each injected issue. This is useful because the agent should learn to distinguish direct evidence from weak hints.

In [ ]:
examples = {
    "srm": generator.generate("exp_srm", failure_modes=["srm"]),
    "imbalance": generator.generate("exp_imbalance", failure_modes=["imbalance"]),
    "low_power": generator.generate("exp_low_power", failure_modes=["low_power"]),
    "outliers": generator.generate("exp_outliers", failure_modes=["outliers"]),
    "simpsons_paradox": generator.generate("exp_simpsons", failure_modes=["simpsons_paradox"]),
}

{name: example.hidden_truth for name, example in examples.items()}

## Visualizing the variance pattern

Outlier-heavy experiments are represented by unusually large post-period standard deviations. This simple comparison helps make that visible.

In [ ]:
import matplotlib.pyplot as plt

labels = ["control_pre", "control_post", "treatment_pre", "treatment_post"]
values = [
    examples["outliers"].control_pre_std,
    examples["outliers"].control_post_std,
    examples["outliers"].treatment_pre_std,
    examples["outliers"].treatment_post_std,
]

plt.figure(figsize=(8, 4))
plt.bar(labels, values, color=["#4C78A8", "#F58518", "#54A24B", "#E45756"])
plt.title("Variance profile for an outlier-heavy synthetic experiment")
plt.ylabel("Standard deviation")
plt.xticks(rotation=15)
plt.show()

## Why summaries are enough here

The goal of the project is not to recover a perfect causal estimate from raw logs. It is to demonstrate how an LLM agent can reason about common experiment pathologies, critique itself, and improve its final answer after reflection.

Summary-level data is enough for that educational objective because the most important patterns are still visible:

- group allocation irregularities
- pre/post shifts
- noisy effects
- variance spikes
- segment-mix reversals